In [11]:
import pymysql

In [2]:
import pandas as pd
from datetime import datetime

#step 1 : Loading the data 
df = pd.read_csv(r'C:\Users\surya\Downloads\traffic_stops - traffic_stops_with_vehicle_number.csv')
df.head()

C:\Users\surya\AppData\Local\Temp\ipykernel_25480\3442131631.py:5: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'C:\Users\surya\Downloads\traffic_stops - traffic_stops_with_vehicle_number.csv')


,stop_date,stop_time,country_name,driver_gender,driver_age_raw,driver_age,driver_race,violation_raw,violation,search_conducted,search_type,stop_outcome,is_arrested,stop_duration,drugs_related_stop,vehicle_number
0,2020-01-01,0:00:00,Canada,M,59,19,Asian,Drunk Driving,Speeding,True,Vehicle Search,Ticket,True,16-30 Min,True,UP76DY3473
1,2020-01-01,0:01:00,India,M,35,58,Other,Other,Other,False,Vehicle Search,Arrest,True,16-30 Min,True,RJ83PZ4441
2,2020-01-01,0:02:00,USA,M,26,76,Black,Signal Violation,Speeding,False,Frisk,Ticket,True,16-30 Min,True,RJ32OM7264
3,2020-01-01,0:03:00,Canada,M,26,76,Black,Speeding,DUI,True,Frisk,Warning,False,0-15 Min,True,RJ76TI3807
4,2020-01-01,0:04:00,Canada,M,62,75,Other,Speeding,Other,False,Vehicle Search,Arrest,True,16-30 Min,False,WB63BB8305


In [4]:
df['stop_duration'].unique()

array(['16-30 Min', '0-15 Min', '30+ Min'], dtype=object)

In [5]:
duration_mapping = {
    '0-15 Min': 7.5,
    '16-30 Min': 22.5,
    '30+ Min': 30
}
df['timing'] = df['stop_duration'].map(duration_mapping)

In [6]:
# step 2: data cleaning 
#Dropping columns with all missing values

df.dropna(axis=1, how = 'all' , inplace = True )   # here axis=1 means drop columns and how = 'all' means drop the column only if all values are NaN.

# Fill NaNs with Suitable values
df.fillna({
    'driver_age' : df['driver_age'].median(),
    'search_type':'None',
    'stop_duration':'Unknown',
    'violatio':'Unknown',
    'stop_outcome':'Unknown',
},inplace=True)

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65538 entries, 0 to 65537
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   stop_date           65538 non-null  object 
 1   stop_time           65538 non-null  object 
 2   country_name        65538 non-null  object 
 3   driver_gender       65538 non-null  object 
 4   driver_age_raw      65538 non-null  int64  
 5   driver_age          65538 non-null  int64  
 6   driver_race         65538 non-null  object 
 7   violation_raw       65538 non-null  object 
 8   violation           65538 non-null  object 
 9   search_conducted    65538 non-null  bool   
 10  search_type         65538 non-null  object 
 11  stop_outcome        65538 non-null  object 
 12  is_arrested         65538 non-null  bool   
 13  stop_duration       65538 non-null  object 
 14  drugs_related_stop  65538 non-null  bool   
 15  vehicle_number      65538 non-null  object 
 16  timi

In [9]:
#Convert Date and time to timestamp 
df['timestamp']=pd.to_datetime(df['stop_date']+' '+df['stop_time'])

df['stop_date'] = pd.to_datetime(df['stop_date'], format="%Y-%m-%d")
df['stop_time'] = pd.to_datetime(df['stop_time'], format="%H:%M:%S")


In [14]:
#Connecting to mysql

conn = pymysql.connect(
    host="localhost",
    user="root",
    password="1234"
)

cursor = conn.cursor()
cursor.execute("create database SecureCheck01")   #Creating a DB named SecureCheck

1

In [34]:
cursor.execute("Use securecheck01")  #Using the Securecheck01 DB


0

In [35]:
cursor.execute('DROP TABLE IF EXISTS police_logs')


0

In [36]:
cursor.execute("""CREATE TABLE IF NOT EXISTS police_logs(
   id int AUTO_INCREMENT PRIMARY KEY , 
   stop_date DATE , 
   stop_time TIME ,
   country_name varchar(20), 
   driver_gender varchar(10),
   driver_age int ,
   driver_race varchar(20),
   violation varchar(20),
   search_conducted BOOLEAN,
   search_type varchar(20),
   stop_outcome varchar(20),
   is_arrested BOOLEAN ,
   stop_duration varchar(50) , 
   drugs_related_stop BOOLEAN , 
   vehicle_number varchar(50),
   timestamp DATETIME ,
   timing float
)""")

0

In [17]:
df.dtypes

stop_date             datetime64[ns]
stop_time             datetime64[ns]
country_name                  object
driver_gender                 object
driver_age_raw                 int64
driver_age                     int64
driver_race                   object
violation_raw                 object
violation                     object
search_conducted                bool
search_type                   object
stop_outcome                  object
is_arrested                     bool
stop_duration                 object
drugs_related_stop              bool
vehicle_number                object
timing                       float64
timestamp             datetime64[ns]
dtype: object

In [18]:
df.isnull().sum().sum()

np.int64(0)

In [19]:
df.columns

Index(['stop_date', 'stop_time', 'country_name', 'driver_gender',
       'driver_age_raw', 'driver_age', 'driver_race', 'violation_raw',
       'violation', 'search_conducted', 'search_type', 'stop_outcome',
       'is_arrested', 'stop_duration', 'drugs_related_stop', 'vehicle_number',
       'timing', 'timestamp'],
      dtype='object')

In [37]:
#Filter and rename columns to match SQL table
df_filtered = df[['stop_date', 'stop_time', 'country_name', 'driver_gender',
                  'driver_age', 'driver_race', 'violation', 'search_conducted',
                  'search_type', 'stop_outcome', 'is_arrested', 'stop_duration',
                  'drugs_related_stop','vehicle_number','timing','timestamp']]


In [38]:
insert_query = """
    INSERT INTO police_logs (
        stop_date,
        stop_time,
        country_name,
        driver_gender,
        driver_age,
        driver_race,
        violation,
        search_conducted,
        search_type,
        stop_outcome,
        is_arrested,
        stop_duration,
        drugs_related_stop,
        vehicle_number,
        timing,
        timestamp
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""


In [39]:
records = [tuple(x) for x in df_filtered.itertuples(index=False, name=None)]

cursor.executemany(insert_query, records)
conn.commit()

print("✅ Data inserted successfully!")


✅ Data inserted successfully!


In [40]:
df.dtypes

stop_date             datetime64[ns]
stop_time             datetime64[ns]
country_name                  object
driver_gender                 object
driver_age_raw                 int64
driver_age                     int64
driver_race                   object
violation_raw                 object
violation                     object
search_conducted                bool
search_type                   object
stop_outcome                  object
is_arrested                     bool
stop_duration                 object
drugs_related_stop              bool
vehicle_number                object
timing                       float64
timestamp             datetime64[ns]
dtype: object